## POST-SUBMISSION FIX — FAO Pivot Correction

**Bug found during Phase 3 audit:**
The original Step 6 in Data_Cleaning_Group_6.ipynb called pivot_table()
on the entire FAO dataframe without filtering by Element first.
This caused all three elements (Food supply kg, Protein g, Fat g) to be
averaged together into one column per year, producing values like
4.25 kg/cap/yr for wheat (correct value: 114.12 kg/cap/yr).

**Impact:** EDA 2 (poverty vs food supply) and EDA 4 (Gini vs food supply)
both used these incorrect numbers. All FAO-based findings must be recomputed.

**Fix applied below:** Each element is filtered separately before pivoting,
then the three separate wide tables are concatenated into one correct file.


In [2]:
import pandas as pd
import numpy as np

# ── Load the raw FAO file (same path as original notebook) ──
DATA = '/kaggle/input/datasets/kinzaemannn/datasetssss/'
fao_raw = pd.read_csv(DATA + 'FAO food supply pakistan.csv')

# ── First: check what elements exist ──────────────────────
print('Elements in raw FAO file:')
print(fao_raw['Element'].unique())
print(f'Total rows: {len(fao_raw)}')



Elements in raw FAO file:
['Food supply quantity (kg/capita/yr)'
 'Protein supply quantity (g/capita/day)'
 'Fat supply quantity (g/capita/day)']
Total rows: 3615


# ─────────────────────────────────────────────────────────────
* THE FIX: Filter FIRST, then pivot SEPARATELY for each element
* Original bug: pivot_table was called on all 3 elements at once,
* causing pandas to average kg, g-protein, and g-fat together.
* That is like averaging apples, oranges, and bananas — meaningless.
# ─────────────────────────────────────────────────────────────


In [3]:
# Step A: Food supply quantity (kg/capita/yr)
food_q = fao_raw[fao_raw['Element'] == 'Food supply quantity (kg/capita/yr)'].copy()
food_pivot = food_q.pivot_table(
    index   = 'Item',
    columns = 'Year',
    values  = 'Value',
    aggfunc = 'first'   # 'first' because each Item-Year combo is unique
).reset_index()
food_pivot.columns.name = None
food_pivot.insert(1, 'Element', 'Food supply quantity (kg/capita/yr)')
food_pivot.insert(2, 'Unit', 'kg/cap')


In [4]:
# Step B: Protein supply quantity (g/capita/day)
protein = fao_raw[fao_raw['Element'] == 'Protein supply quantity (g/capita/day)'].copy()
prot_pivot = protein.pivot_table(
    index   = 'Item',
    columns = 'Year',
    values  = 'Value',
    aggfunc = 'first'
).reset_index()
prot_pivot.columns.name = None
prot_pivot.insert(1, 'Element', 'Protein supply quantity (g/capita/day)')
prot_pivot.insert(2, 'Unit', 'g/cap/d')



In [5]:
# Step C: Fat supply quantity (g/capita/day)
fat = fao_raw[fao_raw['Element'] == 'Fat supply quantity (g/capita/day)'].copy()
fat_pivot = fat.pivot_table(
    index   = 'Item',
    columns = 'Year',
    values  = 'Value',
    aggfunc = 'first'
).reset_index()
fat_pivot.columns.name = None
fat_pivot.insert(1, 'Element', 'Fat supply quantity (g/capita/day)')
fat_pivot.insert(2, 'Unit', 'g/cap/d')


In [6]:
# Step D: Combine all three into one wide table
fao_wide_correct = pd.concat([food_pivot, prot_pivot, fat_pivot], ignore_index=True)

# ── Verification: confirm wheat value is correct ──────────
wheat_check = fao_wide_correct[
    (fao_wide_correct['Item'] == 'Wheat and products') &
    (fao_wide_correct['Element'] == 'Food supply quantity (kg/capita/yr)')
]
print('\n=== VERIFICATION: Wheat food supply (should be ~114 in 2010) ===')
print(wheat_check[['Item', 'Element', 2010, 2015, 2022]].to_string())



=== VERIFICATION: Wheat food supply (should be ~114 in 2010) ===
                  Item                              Element    2010   2015    2022
85  Wheat and products  Food supply quantity (kg/capita/yr)  114.12  98.68  108.69


**── Verification: confirm total protein is sensible ───────**

In [7]:
# ── Verification: confirm total protein is sensible ───────
total_protein_2022 = fao_wide_correct[
    fao_wide_correct['Element'] == 'Protein supply quantity (g/capita/day)'
][2022].sum()
print(f'\nTotal protein 2022: {total_protein_2022:.2f} g/cap/day')
print('(Should be ~70-75. WHO minimum is 50-60. Your old broken file showed 7.8)')

print(f'\nFAO wide shape: {fao_wide_correct.shape}')
print('(Should be around 264 rows x 17 columns)')



Total protein 2022: 72.78 g/cap/day
(Should be ~70-75. WHO minimum is 50-60. Your old broken file showed 7.8)

FAO wide shape: (262, 17)
(Should be around 264 rows x 17 columns)


In [8]:
# Round to 2 decimal places (same as original cleaning notebook)
year_cols = [c for c in fao_wide_correct.columns if isinstance(c, int)]
fao_wide_correct[year_cols] = fao_wide_correct[year_cols].round(2)

# Save
fao_wide_correct.to_csv('fao_food_supply_clean_FIXED.csv', index=False)
print('Saved: fao_food_supply_clean_FIXED.csv')
print(f'Shape: {fao_wide_correct.shape}')


Saved: fao_food_supply_clean_FIXED.csv
Shape: (262, 17)


## POST-SUBMISSION FIX — Gini Index Fabrication Disclosed

**Bug found during Phase 3 audit:**
worldbank_clean.csv contains Gini index values for 2019-2024 that
increase by exactly 0.65 points per year — a perfect arithmetic sequence.
Real household survey data never shows this pattern.

Pakistan's World Bank surveys were last conducted in 2018.
The 2019-2024 values appear to be linear extrapolations that were
never disclosed as estimates. This means EDA 4 (Gini vs food supply)
computed a correlation between two datasets where one had fabricated values.

**Fix applied below:**
1. Only real survey observation years are retained for analysis.
2. The worldbank file is documented with an 'Is_Real_Survey' flag column.
3. EDA 4 is rewritten using only confirmed real data points.


In [9]:
import pandas as pd

# Load the world bank clean file
DATA_CLEAN = '/kaggle/input/datasets/kinzaemannn/datasetssss/'
wb = pd.read_csv(DATA_CLEAN + 'worldbank_clean.csv')

print('=== Original Gini values 2018-2025 (showing the fabrication) ===')
print(wb[wb['Year'] >= 2018][['Year', 'Gini_index']].to_string())


=== Original Gini values 2018-2025 (showing the fabrication) ===
    Year  Gini_index
17  2018       29.60
18  2019       30.25
19  2020       30.90
20  2021       31.55
21  2022       32.20
22  2023       32.85
23  2024       33.50
24  2025       33.50


# ─────────────────────────────────────────────────────────────
* THE FIX: Add a flag column marking which years have REAL survey data
* Real Pakistan World Bank household survey years:
* 2001, 2002, 2004, 2005, 2007, 2010, 2011, 2013, 2015, 2018
* All other years are interpolated/forward-filled — not real observations
# ─────────────────────────────────────────────────────────────


In [10]:
real_survey_years = [2001, 2002, 2004, 2005, 2007, 2010, 2011, 2013, 2015, 2018]

# Add a column: 1 = real observed survey year, 0 = estimated/interpolated
wb['Is_Real_Survey'] = wb['Year'].isin(real_survey_years).astype(int)

print('\n=== Flagged dataset (Is_Real_Survey = 1 means actual data) ===')
print(wb[['Year', 'Gini_index', 'Poverty_headcount_national_pct',
         'Is_Real_Survey']].to_string())



=== Flagged dataset (Is_Real_Survey = 1 means actual data) ===
    Year  Gini_index  Poverty_headcount_national_pct  Is_Real_Survey
0   2001       28.70                           64.30               1
1   2002       29.43                           60.10               1
2   2003       30.17                           55.90               0
3   2004       30.90                           51.70               1
4   2005       31.30                           50.40               1
5   2006       30.50                           47.25               0
6   2007       29.70                           44.10               1
7   2008       29.40                           41.67               0
8   2009       29.10                           39.23               0
9   2010       28.80                           36.80               1
10  2011       29.70                           36.30               1
11  2012       29.60                           32.90               0
12  2013       29.50                   

In [11]:
# Also create a filtered version with ONLY real survey years
# (use this for any regression or correlation analysis)
wb_real_only = wb[wb['Is_Real_Survey'] == 1].copy()
print(f'\nReal survey years only: {list(wb_real_only["Year"])}')
print(f'Real Gini values: {list(wb_real_only["Gini_index"])}')



Real survey years only: [2001, 2002, 2004, 2005, 2007, 2010, 2011, 2013, 2015, 2018]
Real Gini values: [28.7, 29.43, 30.9, 31.3, 29.7, 28.8, 29.7, 29.5, 31.3, 29.6]


In [12]:
# Save both versions
wb.to_csv('worldbank_clean_FIXED.csv', index=False)
wb_real_only.to_csv('worldbank_real_surveys_only.csv', index=False)
print('\nSaved: worldbank_clean_FIXED.csv (all years with flag)')
print('Saved: worldbank_real_surveys_only.csv (only real data)')



Saved: worldbank_clean_FIXED.csv (all years with flag)
Saved: worldbank_real_surveys_only.csv (only real data)


## POST-SUBMISSION FIX — SPI Item Alias Resolution

**Bug found during Phase 3 audit:**
The saved spi-price-clean_F.csv still contains duplicate item names due to
source database field-length truncation. For example:
  - 'Vegetable Ghee Dalda/Habib 2.5' (1680 rows)
  - 'Vegetable Ghee Dalda/Habib Or' (1648 rows)
  - 'Vegetable Ghee Dalda/Habib Or O' (32 rows)
These all refer to the same product but groupby treats them as 3 items.

The cleaning notebook reported doing this fix, but the saved CSV shows
the fix was only applied inside individual EDA code cells, not in the
cleaning step. The cleaned output file must contain the resolved names.

**Fix:** Apply the complete alias dictionary to the clean SPI file and
save a new version with resolved item names.


In [13]:
import pandas as pd

# Load the clean SPI file
DATA_CLEAN = '/kaggle/input/datasets/kinzaemannn/datasetssss/'
spi = pd.read_csv(DATA_CLEAN + 'spi-price-clean F.csv', parse_dates=['Date'])

print(f'Items BEFORE alias fix: {spi["Item"].nunique()}')
print('Duplicate variants found:')
print(spi['Item'].value_counts().head(20).to_string())


Items BEFORE alias fix: 60
Duplicate variants found:
Item
Bananas (Kela) Local                1680
Beef With Bone (Average Quality)    1680
Bread Plain (Small Size)            1680
Chicken Farm Broiler (Live)         1680
Chilies Powder National 200 Gm      1680
Cigarettes Capstan 20'S Packet      1680
Cooked Beef At Average Hotel        1680
Cooked Daal At Average Hotel        1680
Eggs Hen (Farm)                     1680
Curd (Dahi) Loose                   1680
Energy Saver Philips 14 Watt        1680
Firewood Whole                      1680
Ladies Sandal Bata                  1680
Lawn Printed Gul Ahmed/Al Karam     1680
Gents Sandal Bata                   1680
Garlic (Lehsun)                     1680
Hi-Speed Diesel                     1680
Gur (Average Quality)               1680
Georgette (Average Quality)         1680
Gents Sponge Chappal Bata           1680


# ─────────────────────────────────────────────────────────────
* THE FIX: Complete alias dictionary
* Every truncated variant maps to one canonical full name.
* Rule: always pick the LONGEST (most complete) version as canonical.
# ─────────────────────────────────────────────────────────────


In [14]:
alias_map = {
    # Cooking Oil (2 variants → 1)
    'Cooking Oil Dalda Or Other Simila'   : 'Cooking Oil Dalda Or Other Similar',

    # Powdered Milk (2 variants → 1)
    'Powdered Milk Nido 390 Gm Polyb'     : 'Powdered Milk Nido 390 Gm Polyba',

    # Rice Basmati (2 variants → 1)
    'Rice Basmati Broken (Average Qua'    : 'Rice Basmati Broken (Average Qual)',

    # Long Cloth (2 variants → 1)
    'Long Cloth 57" Gul Ahmed/Al Kara'   : 'Long Cloth 57" Gul Ahmed/Al Karam',

    # Vegetable Ghee (3 variants → 1)
    'Vegetable Ghee Dalda/Habib 2.5'      : 'Vegetable Ghee Dalda/Habib Or Other',
    'Vegetable Ghee Dalda/Habib Or'        : 'Vegetable Ghee Dalda/Habib Or Other',
    'Vegetable Ghee Dalda/Habib Or O'      : 'Vegetable Ghee Dalda/Habib Or Other',

    # Tea Lipton (2 variants → 1)
    'Tea Lipton Yellow Label 190 Gm Pa'    : 'Tea Lipton Yellow Label 190 Gm Pack',
    'Tea Lipton Yellow Label 190 Gm Pac'   : 'Tea Lipton Yellow Label 190 Gm Pack',

    # Salt (2 variants → 1)
    'Salt Powdered (National/Shan) 8'      : 'Salt Powdered (National/Shan)',

    # Electricity (2 variants → 1)
    'Electricity Charges For Q1*'          : 'Electricity Charges For Q1',
}

# Apply the alias map to the Item column
spi['Item'] = spi['Item'].replace(alias_map)

print(f'\nItems AFTER alias fix: {spi["Item"].nunique()}')
print('(Should drop from 60 to about 52)')



Items AFTER alias fix: 52
(Should drop from 60 to about 52)


In [15]:
# ── Also add Province column (we need this for Part 3 EDA) ─
city_to_province = {
    'Islamabad'  : 'ICT',
    'Rawalpindi' : 'Punjab',
    'Lahore'     : 'Punjab',
    'Faisalabad' : 'Punjab',
    'Gujranwala' : 'Punjab',
    'Multan'     : 'Punjab',
    'Bahawalpur' : 'Punjab',
    'Sargodha'   : 'Punjab',
    'Sialkot'    : 'Punjab',
    'Karachi'    : 'Sindh',
    'Hyderabad'  : 'Sindh',
    'Sukkur'     : 'Sindh',
    'Larkana'    : 'Sindh',
    'Peshawar'   : 'KPK',
    'Bannu'      : 'KPK',
    'Quetta'     : 'Balochistan',
}
spi['Province'] = spi['City'].map(city_to_province)
print('\nProvince column added:')
print(spi['Province'].value_counts().to_string())




Province column added:
Province
Punjab         42527
Sindh          21320
KPK            10710
ICT             5355
Balochistan     5355


**── Flag Quetta Wheat Flour as unreliable** 
* Quetta Wheat Flour Bag shows a constant price of PKR 111.3 across all dates — clearly a data recording error, not a real price.
* We flag it so analysts know to exclude it, but we do NOT delete other Quetta items which are reliable.


In [16]:
spi['Data_Reliable'] = True   # Start: all rows are reliable
spi.loc[
    (spi['City'] == 'Quetta') & (spi['Item'] == 'Wheat Flour Bag'),
    'Data_Reliable'
] = False

flagged = spi[spi['Data_Reliable'] == False]
print(f'\nQuetta Wheat Flour rows flagged as unreliable: {len(flagged)}')




Quetta Wheat Flour rows flagged as unreliable: 105


In [17]:
# Save the fixed file
spi.to_csv('spi_prices_clean_FIXED.csv', index=False)
print('\nSaved: spi_prices_clean_FIXED.csv')
print(f'Final shape: {spi.shape}')
print(f'Columns: {list(spi.columns)}')



Saved: spi_prices_clean_FIXED.csv
Final shape: (85267, 12)
Columns: ['Date', 'Year', 'Month', 'Month_Name', 'City', 'Item', 'Unit', 'Min_Price', 'Avg_Price', 'Max_Price', 'Province', 'Data_Reliable']
